In [15]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import html

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/win/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/win/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [16]:
def create_stem_cache(corpus):
    unique_words = set()
    for text in corpus:
        text = str(text).lower()
        text = re.sub(r'[^a-zà-ÿ\s]', ' ', text)
        unique_words.update(text.split())
    
    stem_cache = {}
    ps = PorterStemmer()
    for w in unique_words:
        stem_cache[w] = ps.stem(w)
    return stem_cache

In [17]:
class CustomPreprocessor:
    def __init__(self, stop_dict, stem_cache):
        self.stop_dict = stop_dict
        self.stem_cache = stem_cache

    def __call__(self, s):
        s = str(s).lower()
        s = re.sub(r'[^a-zà-ÿ\s]', ' ', s)
        tokens = s.split()
        tokens = [w for w in tokens if w not in self.stop_dict and len(w) > 2]
        tokens = [self.stem_cache.get(w, w) for w in tokens]
        return ' '.join(tokens)

In [18]:
class RecipeSearchEngine:
    def __init__(self, vectorizer, data_norm, df):
        self.vectorizer = vectorizer
        self.data_norm = data_norm
        self.df = df

    def search(self, query, top_k=5):
        query_vec = self.vectorizer.transform([query])
        scores = self.data_norm.dot(query_vec.T).toarray().flatten()
        rank = np.argsort(scores)[::-1]
        
        results = self.df.iloc[rank[:top_k]].copy()
        results['Score'] = scores[rank[:top_k]]
        return results

In [19]:
df = pd.read_csv('../data/raw/recipes.csv')
df["Name"] = df["Name"].apply(html.unescape)
df["Description"] = df["Description"].astype(str).apply(html.unescape)
df['SearchCorpus'] = df['Name'] + ' ' + df['RecipeIngredientParts'] + ' ' + df['RecipeInstructions']

In [20]:
stop_dict = set(stopwords.words('english'))
stem_cache = create_stem_cache(df['SearchCorpus'])
my_custom_preprocessor = CustomPreprocessor(stop_dict, stem_cache)

In [21]:
tfidf_vectorizer = TfidfVectorizer(preprocessor=my_custom_preprocessor, use_idf=True)
data_norm = tfidf_vectorizer.fit_transform(df['SearchCorpus'])

In [22]:
searcher = RecipeSearchEngine(tfidf_vectorizer, data_norm, df[['Name']])
with open('../resources/recipe_search_engine.pkl', 'wb') as f:
    pickle.dump(searcher, f)

In [23]:
with open('../resources/recipe_search_engine.pkl', 'rb') as f:
    searcher = pickle.load(f)
results = searcher.search("eggs mix together")

display(results[['Name', 'Score']])

,Name,Score
119150,Scrambled Eggs for 100,0.499159
256565,Egg Salad,0.494592
51378,Cheesecake Lemon Bars,0.482161
206745,Easy Chicken Tenders,0.472361
149049,Deviled Eggs,0.468281
